In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# =========================
# 1. Load Data
# =========================
df = pd.read_csv("big_real_estate_dataset.csv")

# =========================
# 2. Preprocessing
# =========================

# تحويل value_impact لرقم
df["value_impact"] = df["value_impact"].astype(float)

# Encode categorical
label_encoders = {}
categorical_cols = [
    "location_type",
    "current_zone",
    "population_density",
    "avg_income",
    "permit_history",
    "expected_development"
]

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# =========================
# 3. Split Features / Target
# =========================

X = df.drop("expected_development", axis=1)
y = df["expected_development"]

# =========================
# 4. Train / Test Split
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 5. Model (قوي)
# =========================

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

# =========================
# 6. Evaluation
# =========================

y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("🔥 Accuracy:", acc)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# =========================
# 7. Prediction جديد
# =========================

sample = pd.DataFrame([{
    "area_size": 5000,
    "location_type": label_encoders["location_type"].transform(["urban"])[0],
    "distance_to_city_center": 12,
    "distance_to_main_road": 0.5,
    "nearby_projects": 5,
    "price_per_meter": 16000,
    "current_zone": label_encoders["current_zone"].transform(["commercial"])[0],
    "population_density": label_encoders["population_density"].transform(["high"])[0],
    "avg_income": label_encoders["avg_income"].transform(["high"])[0],
    "permit_history": label_encoders["permit_history"].transform(["commercial"])[0],
    "value_impact": 7
}])

pred = model.predict(sample)
pred_label = label_encoders["expected_development"].inverse_transform(pred)

# احتمالات (مهم جدًا للديمو)
probs = model.predict_proba(sample)

print("\n🎯 Prediction:")
print("Expected Development:", pred_label[0])

print("\n📊 Probabilities:")
for i, cls in enumerate(label_encoders["expected_development"].classes_):
    print(cls, ":", round(probs[0][i]*100, 2), "%")

🔥 Accuracy: 0.995

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.80      0.89        10
           1       1.00      1.00      1.00        98
           2       1.00      1.00      1.00        46
           3       0.96      1.00      0.98        51
           4       1.00      1.00      1.00        55
           5       1.00      1.00      1.00       140

    accuracy                           0.99       400
   macro avg       0.99      0.97      0.98       400
weighted avg       1.00      0.99      0.99       400


🎯 Prediction:
Expected Development: shopping_center

📊 Probabilities:
mall : 31.73 %
residential_compound : 1.0 %
school : 19.78 %
shopping_center : 42.56 %
towers : 4.2 %
warehouse : 0.73 %


In [9]:
import joblib

joblib.dump(model, "model.pkl")
joblib.dump(label_encoders, "encoders.pkl")

['encoders.pkl']